# EDA — producción agrícola y clima (regional y por cultivo)

Exploración **descriptiva** de `dataset_regional.csv` (región × piso × mes) y `dataset_integrado.csv` (región × cultivo × mes), ambos restringidos a las combinaciones más representativas.

**Objetivos (alineados al paper):**
1. Caracterizar volumen y estacionalidad productiva por unidad territorial.
2. Describir perfiles climáticos por piso ecológico.
3. Explorar co-movimientos clima–producción (Pearson, sin inferencia causal), a nivel regional y de cultivo.
4. Documentar eventos extremos: sequía Puno 2021–2022 y El Niño costero 2023–2024.

**Limitaciones explícitas:**
- Correlación ≠ causalidad; sin desestacionalización ni corrección Benjamini–Hochberg.
- Cultivos del mismo (región, piso, distrito) comparten clima idéntico.
- Producción en toneladas (volumen), no rendimiento t/ha.

**Unidades:** `radiacion_solar` MJ/m²/día; `precipitacion` mm/día; `humedad_suelo` índice 0–1.

**Downstream:** estos hallazgos contextualizan `06_clustering_final.ipynb`.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif ROOT.name == "SCRIPTS":
    ROOT = ROOT.parent
elif not (ROOT / "OUTPUTS").exists() and (ROOT.parent / "OUTPUTS").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "SCRIPTS"))
from viz_style import REGION_COLORS, REGION_ORDER, PLOTLY_TEMPLATE

px.defaults.template = PLOTLY_TEMPLATE


def titulo_centrado(texto, size=18):
    """Titulo de figura centrado y en negrita, mas prominente que los ejes."""
    return dict(text=f"<b>{texto}</b>", x=0.5, xanchor="center", font=dict(size=size))


def hex_a_rgba(hexcolor, alpha=0.75):
    """Convierte un hex (#RRGGBB) a rgba(...) con la opacidad indicada."""
    hexcolor = hexcolor.lstrip("#")
    r, g, b = int(hexcolor[0:2], 16), int(hexcolor[2:4], 16), int(hexcolor[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

RUTA_OUTPUT = ROOT / "OUTPUTS"
RUTA_FIGURES = RUTA_OUTPUT / "figures"
RUTA_FIGURES.mkdir(parents=True, exist_ok=True)
DATASET_REGIONAL = RUTA_OUTPUT / "dataset_regional.csv"
DATASET_INTEGRADO = RUTA_OUTPUT / "dataset_integrado.csv"

if not DATASET_REGIONAL.exists():
    raise FileNotFoundError(f"No se encontró {DATASET_REGIONAL}. Ejecutar primero pipeline_integrado.py")

df = pd.read_csv(DATASET_REGIONAL)
df["unidad"] = df["region"] + " | " + df["piso_ecologico"] + " (" + df["distrito"] + ")"
df["fecha"] = pd.to_datetime(df["anio"].astype(str) + "-" + df["numero_mes"].astype(str).str.zfill(2) + "-01")
df_cultivo = pd.read_csv(DATASET_INTEGRADO)

CLIMA_EDA = ["temp_promedio", "precipitacion", "humedad_relativa", "radiacion_solar", "humedad_suelo"]
MESES_ORDEN = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
               "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
MESES_ABREV = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
df["mes"] = pd.Categorical(df["mes"], categories=MESES_ORDEN, ordered=True)
df_cultivo["mes"] = pd.Categorical(df_cultivo["mes"], categories=MESES_ORDEN, ordered=True)


def guardar(fig, nombre):
    """Exporta cada figura como PNG estático (informe) y HTML interactivo (exploración).

    write_image no hereda fig.layout.width/height por defecto: hay que pasarlos explícitos.
    """
    fig.write_image(RUTA_FIGURES / f"{nombre}.png", scale=2, width=fig.layout.width, height=fig.layout.height)
    fig.write_html(RUTA_FIGURES / f"{nombre}.html", include_plotlyjs="cdn")


print(f"Filas: {len(df):,} | Unidades: {df['unidad'].nunique()} | Regiones: {df['region'].nunique()}")
print(f"NaN produccion_piso_ton: {df['produccion_piso_ton'].isna().sum()}")
df.head(3)


Filas: 2,448 | Unidades: 34 | Regiones: 6
NaN produccion_piso_ton: 157


,region,piso_ecologico,distrito,anio,numero_mes,mes,num_cultivos,produccion_piso_ton,temp_promedio,temp_maxima,...,humedad_relativa,radiacion_solar,velocidad_viento,presion_atmosferica,humedad_suelo,temp_superficie,punto_rocio,humedad_especifica,unidad,fecha
0,Ica,costa,Chincha Alta,2020,1,Enero,26,18468.379,21.96,27.55,...,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41,Ica | costa (Chincha Alta),2020-01-01
1,Ica,costa,Chincha Alta,2020,2,Febrero,26,11909.728,22.85,27.98,...,79.22,23.67,2.84,95.77,0.33,24.72,18.86,14.31,Ica | costa (Chincha Alta),2020-02-01
2,Ica,costa,Chincha Alta,2020,3,Marzo,26,9570.669,22.62,27.86,...,78.16,22.84,2.70,95.82,0.33,24.34,18.41,13.90,Ica | costa (Chincha Alta),2020-03-01


# Sección 1 — Análisis regional y por cultivo

## 1. Volumen productivo por unidad

Ranking de las 14 unidades (región × piso × distrito) según producción acumulada 2020–2025.

In [2]:
vol_unidad = (
    df.groupby(["region", "unidad"], observed=True)["produccion_piso_ton"]
    .sum(min_count=1)
    .reset_index(name="produccion_total_ton")
    .sort_values("produccion_total_ton")
)

fig = px.bar(
    vol_unidad, x="produccion_total_ton", y="unidad", color="region", orientation="h",
    color_discrete_map=REGION_COLORS, category_orders={"region": REGION_ORDER},
    labels={"produccion_total_ton": "Toneladas", "unidad": ""},
)
fig.update_layout(
    title=titulo_centrado("Producción acumulada 2020–2025 por unidad (combinaciones más representativas)"),
    height=580, margin=dict(t=90),
)
guardar(fig, "eda_regional_volumen_unidad")
fig.show()


## 2. Producción mensual por región

Serie agregada de todas las unidades. Es la figura "ancla" que la Sección 2 (eventos extremos) profundiza.

In [3]:
prod_mensual = (
    df.groupby(["region", "fecha"], observed=True)["produccion_piso_ton"]
    .sum(min_count=1)
    .reset_index()
)

fig = px.line(
    prod_mensual, x="fecha", y="produccion_piso_ton", color="region",
    color_discrete_map=REGION_COLORS, category_orders={"region": REGION_ORDER},
    labels={"produccion_piso_ton": "Toneladas", "fecha": ""},
)
fig.update_layout(title=titulo_centrado("Producción mensual agregada por región"), margin=dict(t=90))
guardar(fig, "eda_regional_produccion_mensual")
fig.show()


### Patrón estacional promedio por región y piso ecológico

Promedio mensual 2020–2025 de producción por unidad (región × piso), para comparar la
**forma del calendario productivo**: ciclo único y concentrado (altiplano/puna de Puno)
frente a producción uniforme todo el año (costa agroexportadora).

In [4]:
patron_estacional = (
    df.groupby(["region", "piso_ecologico", "mes"], observed=True)["produccion_piso_ton"]
    .mean()
    .reset_index()
)

fig = px.line(
    patron_estacional, x="mes", y="produccion_piso_ton", color="region", line_dash="piso_ecologico",
    color_discrete_map=REGION_COLORS, category_orders={"region": REGION_ORDER, "mes": MESES_ORDEN},
    labels={"produccion_piso_ton": "Toneladas (promedio mensual)", "mes": "", "piso_ecologico": "Piso"},
)
fig.update_layout(
    title=titulo_centrado("Patrón estacional promedio por región y piso ecológico"),
    width=1100, height=550, margin=dict(t=90),
    legend=dict(font=dict(size=10)),
)
guardar(fig, "eda_regional_patron_estacional")
fig.show()

## 3. Estacionalidad (mes × año), en toneladas

Heatmap por región, en toneladas (no %). Cada panel usa su **propia escala de color** (no una escala compartida): así Puno no aplasta el contraste de las demás regiones, y cada barra de color de la derecha muestra las toneladas reales de ese panel.

In [5]:
from plotly.subplots import make_subplots

ncols = 3
nrows = -(-len(REGION_ORDER) // ncols)

fig = make_subplots(
    rows=nrows, cols=ncols, subplot_titles=REGION_ORDER,
    horizontal_spacing=0.14, vertical_spacing=0.22,
)

coloraxes = {}
for i, region in enumerate(REGION_ORDER):
    sub = df[df["region"] == region]
    tabla = sub.pivot_table(
        index="mes", columns="anio", values="produccion_piso_ton",
        aggfunc=lambda x: x.sum(min_count=1), observed=True,
    ).reindex(MESES_ORDEN)

    row, col = i // ncols + 1, i % ncols + 1
    eje = "" if i == 0 else str(i + 1)
    xdom = fig.layout[f"xaxis{eje}"].domain
    ydom = fig.layout[f"yaxis{eje}"].domain
    nombre_coloraxis = "coloraxis" if i == 0 else f"coloraxis{i + 1}"

    fig.add_trace(
        go.Heatmap(z=tabla.values, x=tabla.columns, y=MESES_ORDEN, coloraxis=nombre_coloraxis),
        row=row, col=col,
    )
    coloraxes[nombre_coloraxis] = dict(
        colorscale="YlOrRd",
        colorbar=dict(
            x=xdom[1] + 0.02, y=sum(ydom) / 2, len=(ydom[1] - ydom[0]) * 0.9,
            thickness=12, title=dict(text="ton", font=dict(size=9)), tickfont=dict(size=8),
        ),
    )

fig.update_layout(
    title=titulo_centrado("Estacionalidad productiva — toneladas por mes x ano, escala propia por region"),
    width=1500, height=820, margin=dict(t=100),
    **coloraxes,
)
fig.update_xaxes(dtick=1)
guardar(fig, "eda_regional_heatmap_estacionalidad")
fig.show()


### Estacionalidad por cultivo, dentro de cada región

Cada celda es el total de toneladas 2020–2025 de ese cultivo en ese mes (no %). Igual que el heatmap anterior, cada panel de región usa su **propia escala de color**.

In [6]:
cultivo_mes = (
    df_cultivo.groupby(["region", "cultivo", "mes"], observed=True)["produccion_ton"]
    .sum(min_count=1)
    .reset_index()
)

# go.Heatmap por región (no facet de px) para que cada panel solo liste sus propios cultivos
# y tenga su propia escala de color (en toneladas, no %) — mismo enfoque que la fig. anterior
from plotly.subplots import make_subplots

regiones_cultivo = [r for r in REGION_ORDER if r in cultivo_mes["region"].unique()]
ncols = 3
nrows = -(-len(regiones_cultivo) // ncols)

fig = make_subplots(
    rows=nrows, cols=ncols, subplot_titles=regiones_cultivo,
    horizontal_spacing=0.14, vertical_spacing=0.22,
)

coloraxes = {}
for i, region in enumerate(regiones_cultivo):
    sub = cultivo_mes[cultivo_mes["region"] == region]
    cultivos_region = sorted(sub["cultivo"].unique())
    tabla = sub.pivot_table(index="cultivo", columns="mes", values="produccion_ton", aggfunc="sum")
    tabla = tabla.reindex(index=cultivos_region, columns=MESES_ORDEN)

    row, col = i // ncols + 1, i % ncols + 1
    eje = "" if i == 0 else str(i + 1)
    xdom = fig.layout[f"xaxis{eje}"].domain
    ydom = fig.layout[f"yaxis{eje}"].domain
    nombre_coloraxis = "coloraxis" if i == 0 else f"coloraxis{i + 1}"

    fig.add_trace(
        go.Heatmap(z=tabla.values, x=MESES_ABREV, y=cultivos_region, coloraxis=nombre_coloraxis),
        row=row, col=col,
    )
    coloraxes[nombre_coloraxis] = dict(
        colorscale="YlOrRd",
        colorbar=dict(
            x=xdom[1] + 0.02, y=sum(ydom) / 2, len=(ydom[1] - ydom[0]) * 0.9,
            thickness=12, title=dict(text="ton", font=dict(size=9)), tickfont=dict(size=8),
        ),
    )

fig.update_layout(
    title=titulo_centrado("Estacionalidad por cultivo — toneladas por mes, escala propia por region"),
    width=1500, height=950,
    margin=dict(t=110, b=80),
    **coloraxes,
)
fig.update_xaxes(tickfont=dict(size=10))
fig.update_yaxes(tickfont=dict(size=10))
guardar(fig, "eda_regional_heatmap_cultivo_mes")
fig.show()


C:\Users\Joyssie\AppData\Local\Temp\ipykernel_31268\629690462.py:24: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

C:\Users\Joyssie\AppData\Local\Temp\ipykernel_31268\629690462.py:24: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

C:\Users\Joyssie\AppData\Local\Temp\ipykernel_31268\629690462.py:24: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

C:\Users\Joyssie\AppData\Local\Temp\ipykernel_31268\629690462.py:24: FutureWarning:

The default value of observed=False is deprecated and will change to observed=

## 4. Perfil climático y variabilidad térmica por región

Medias 2020–2025 agregadas a nivel región (no por unidad, para evitar saturación visual).

In [7]:
from plotly.subplots import make_subplots

clima_region = (
    df.groupby("region", observed=True)[CLIMA_EDA].mean()
    .reset_index()
    .melt(id_vars="region", var_name="variable", value_name="valor")
)

# Subplot explicito (no facet de px): temp/precip/humedad/radiacion/suelo tienen
# unidades y escalas muy distintas (ej. humedad_relativa ~80 vs precipitacion ~0.5);
# cada panel es una variable con su propio eje x (region) e y (su unidad), para
# que cada mini-grafico se lea solo, con titulo y ejes propios.
NOMBRES_CLIMA = {
    "temp_promedio": "Temp. promedio (°C)",
    "precipitacion": "Precipitación (mm/día)",
    "humedad_relativa": "Humedad relativa (%)",
    "radiacion_solar": "Radiación solar (MJ/m²/día)",
    "humedad_suelo": "Humedad suelo (índice)",
}
clima_region["variable"] = clima_region["variable"].map(NOMBRES_CLIMA)
variables_clima = list(NOMBRES_CLIMA.values())

ncols = 3
nrows = -(-len(variables_clima) // ncols)
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=variables_clima,
                     horizontal_spacing=0.08, vertical_spacing=0.24)
for i, var_nombre in enumerate(variables_clima):
    sub = (clima_region[clima_region["variable"] == var_nombre]
           .set_index("region").reindex(REGION_ORDER))
    row, col = i // ncols + 1, i % ncols + 1
    fig.add_trace(
        go.Bar(x=REGION_ORDER, y=sub["valor"],
               marker_color=[REGION_COLORS[r] for r in REGION_ORDER], showlegend=False),
        row=row, col=col,
    )
fig.update_xaxes(tickangle=0, tickfont=dict(size=9))
fig.update_layout(
    title=titulo_centrado("Perfil climático promedio por región (2020–2025)"),
    width=1300, height=700, margin=dict(t=110),
)
guardar(fig, "eda_regional_perfil_climatico")
fig.show()

# x = piso (sin facet) para no multiplicar paneles y evitar etiquetas superpuestas
fig = px.box(
    df, x="piso_ecologico", y="temp_promedio", color="region", boxmode="group",
    color_discrete_map=REGION_COLORS, category_orders={"region": REGION_ORDER},
    labels={"piso_ecologico": "Piso ecológico", "temp_promedio": "Temperatura (°C)"},
)
# por defecto Plotly rellena las cajas con una version mas tenue del color de linea;
# se fija fillcolor explicito para que tengan el mismo peso visual que las barras
# de los demas graficos (misma paleta REGION_COLORS, ya no solo el contorno).
for trace in fig.data:
    trace.fillcolor = hex_a_rgba(trace.marker.color, 0.75)
    trace.line = dict(color=trace.marker.color, width=1.5)
fig.update_layout(
    title=titulo_centrado("Distribución mensual de temperatura por región y piso"),
    margin=dict(b=120, t=100), width=1300, height=580,
    boxgap=0.25, boxgroupgap=0.12,
)
fig.update_xaxes(tickangle=30)
guardar(fig, "eda_regional_boxplot_temp")
fig.show()

## 5. Correlaciones exploratorias (Pearson) — por unidad regional

Asociación mensual clima–producción **por unidad**. Sin corrección por comparaciones múltiples.

In [8]:
# Pearson producción-clima por unidad (≥12 meses), exploratorio, sin corrección BH
rows = []
for unidad, g in df.groupby("unidad"):
    sub = g.dropna(subset=["produccion_piso_ton"] + CLIMA_EDA)
    if len(sub) < 12:
        continue
    region = g["region"].iloc[0]
    piso = g["piso_ecologico"].iloc[0]
    distrito = g["distrito"].iloc[0]
    for var in CLIMA_EDA:
        r, p = stats.pearsonr(sub["produccion_piso_ton"], sub[var])
        rows.append({
            "unidad": unidad, "region": region, "piso_ecologico": piso,
            "distrito": distrito, "variable_clima": var,
            "n": len(sub), "r": r, "p_valor": p,
        })

corr_regional = pd.DataFrame(rows)
corr_regional.to_csv(RUTA_OUTPUT / "eda_correlaciones_regional.csv", index=False, encoding="utf-8-sig")

top5 = corr_regional.reindex(corr_regional["r"].abs().sort_values(ascending=False).index).head(5)
print("=== Top 5 |r| por unidad (exploratorio, sin BH) ===")
print(top5[["unidad", "variable_clima", "n", "r", "p_valor"]].to_string(index=False))

sub_all = df.dropna(subset=["produccion_piso_ton"] + CLIMA_EDA)
print("\n=== Correlación agregada (pooling de todas las unidades) ===")
for var in CLIMA_EDA:
    r, p = stats.pearsonr(sub_all["produccion_piso_ton"], sub_all[var])
    print(f"{var:20s} r={r:+.3f}  p={p:.2e}  n={len(sub_all)}")


=== Top 5 |r| por unidad (exploratorio, sin BH) ===
                        unidad   variable_clima  n         r      p_valor
Junin | selva_alta (Rio Negro) humedad_relativa 68  0.784593 2.456599e-15
       Ica | costa (El Carmen)  radiacion_solar 68 -0.746432 2.745976e-13
    Ica | costa (Chincha Alta)    temp_promedio 68 -0.689501 7.928814e-11
    Ica | costa (Chincha Alta) humedad_relativa 68 -0.675573 2.616858e-10
Junin | selva_alta (Rio Negro)    precipitacion 68  0.665397 6.015917e-10

=== Correlación agregada (pooling de todas las unidades) ===
temp_promedio        r=-0.044  p=3.43e-02  n=2291
precipitacion        r=-0.028  p=1.87e-01  n=2291
humedad_relativa     r=+0.087  p=3.13e-05  n=2291
radiacion_solar      r=+0.046  p=2.80e-02  n=2291
humedad_suelo        r=+0.047  p=2.58e-02  n=2291


## 6. Correlaciones exploratorias (Pearson) — por cultivo

Mismo análisis que la sección anterior, pero a granularidad cultivo × mes (`dataset_integrado.csv`). Cultivos del mismo piso comparten clima idéntico, así que diferencias entre cultivos de una misma región reflejan sobre todo calendarios de cosecha distintos, no sensibilidades climáticas independientes.

In [9]:
rows = []
for (region, cultivo), g in df_cultivo.groupby(["region", "cultivo"]):
    sub = g.dropna(subset=["produccion_ton"] + CLIMA_EDA)
    if len(sub) < 12:
        continue
    for var in CLIMA_EDA:
        r, p = stats.pearsonr(sub["produccion_ton"], sub[var])
        rows.append({
            "region": region, "cultivo": cultivo, "variable_clima": var,
            "n": len(sub), "r": r, "p_valor": p,
        })

corr_cultivo = pd.DataFrame(rows)
corr_cultivo.to_csv(RUTA_OUTPUT / "eda_correlaciones_por_cultivo.csv", index=False, encoding="utf-8-sig")

top10 = corr_cultivo.reindex(corr_cultivo["r"].abs().sort_values(ascending=False).index).head(10).iloc[::-1]
top10["etiqueta"] = top10["region"] + " — " + top10["cultivo"] + " (" + top10["variable_clima"] + ")"

fig = px.bar(
    top10, x="r", y="etiqueta", color="region", orientation="h",
    color_discrete_map=REGION_COLORS, category_orders={"region": REGION_ORDER},
    labels={"etiqueta": ""},
)
fig.update_layout(
    title=titulo_centrado("Top 10 |r| produccion-clima por (region, cultivo)"),
    margin=dict(l=20, t=90), height=480,
)
guardar(fig, "eda_cultivo_top_correlaciones")
fig.show()


### Correlación clima completo por cultivo dominante

Mismo análisis de Pearson que arriba, pero con las 7 variables climáticas completas
(incluyendo temperatura máxima/mínima) para el cultivo de mayor producción de cada
región, más `alfalfa` (Puno) y `papa` (Junín) citados en el texto.

In [10]:
VARS_CLIMA_COMPLETO = ["temp_promedio", "temp_maxima", "temp_minima", "precipitacion",
                        "humedad_relativa", "radiacion_solar", "humedad_suelo"]
NOMBRES_VARS_CORTO = {
    "temp_promedio": "T media", "temp_maxima": "T max", "temp_minima": "T min",
    "precipitacion": "Precip", "humedad_relativa": "Hum. rel", "radiacion_solar": "Radiación",
    "humedad_suelo": "Hum. suelo",
}
PARES_DOMINANTES = [
    ("La Libertad", "cana_para_azucar"), ("Puno", "avena_forrajera"), ("San Martin", "arroz_cascara"),
    ("Piura", "cana_para_azucar"), ("Ica", "uva"), ("Junin", "pina"), ("Puno", "alfalfa"), ("Junin", "papa"),
]
filas = []
for region, cultivo in PARES_DOMINANTES:
    sub = df_cultivo[(df_cultivo["region"] == region) & (df_cultivo["cultivo"] == cultivo)].dropna(
        subset=["produccion_ton"] + VARS_CLIMA_COMPLETO)
    fila = {"etiqueta": f"{region} | {cultivo}"}
    for v in VARS_CLIMA_COMPLETO:
        r, _ = stats.pearsonr(sub["produccion_ton"], sub[v])
        fila[NOMBRES_VARS_CORTO[v]] = round(r, 2)
    filas.append(fila)
corr_dominantes = pd.DataFrame(filas).set_index("etiqueta")
corr_dominantes.to_csv(RUTA_OUTPUT / "eda_correlaciones_dominantes.csv", encoding="utf-8-sig")

fig = px.imshow(
    corr_dominantes, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
    labels=dict(color="Correlación de Pearson"),
)
fig.update_layout(
    title=titulo_centrado("Correlación clima-producción por cultivo dominante"),
    width=950, height=500, margin=dict(t=90, l=200),
)
guardar(fig, "eda_cultivo_correlacion_dominantes")
fig.show()

# Sección 2 — Eventos climáticos extremos

La fig. 2 (producción mensual por región) muestra dos quiebres visibles: una caída sostenida en Puno (2021→2022) y un repunte de precipitación en la costa norte en 2023 (El Niño costero). Esta sección profundiza en ambos puntos.

## 2.1 Sequía altiplánica — Puno 2021→2022

In [11]:
puno = df[(df["region"] == "Puno") & (df["anio"].between(2020, 2023))]
prod_puno = puno.groupby("anio", observed=True)["produccion_piso_ton"].sum(min_count=1).reset_index()
clima_puno = puno.groupby("anio", observed=True)["humedad_suelo"].mean().reset_index()

p = prod_puno.set_index("anio")["produccion_piso_ton"]
print(f"Cambio producción Puno 2021→2022: {100 * (p.loc[2022] / p.loc[2021] - 1):+.1f}%")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=prod_puno["anio"], y=prod_puno["produccion_piso_ton"], name="Producción",
    line=dict(color=REGION_COLORS["Puno"]),
))
fig.add_trace(go.Scatter(
    x=clima_puno["anio"], y=clima_puno["humedad_suelo"], name="Humedad suelo", yaxis="y2",
    line=dict(color=REGION_COLORS["Puno"], dash="dash"),
))
fig.update_layout(
    title=titulo_centrado("Puno — produccion anual vs humedad de suelo (sequia 2021-2022)"),
    xaxis=dict(title="Año", dtick=1, tickformat="d"),
    yaxis=dict(title="Toneladas"),
    yaxis2=dict(title="Humedad suelo (índice)", overlaying="y", side="right", automargin=True),
    margin=dict(r=120, t=100),
    legend=dict(orientation="h", y=1.18, x=0),
)
guardar(fig, "eda_regional_puno_sequia")
fig.show()


Cambio producción Puno 2021→2022: -8.2%


### Profundización — ¿qué cultivos explican la caída?

In [12]:
puno_cultivo = (
    df_cultivo[(df_cultivo["region"] == "Puno") & (df_cultivo["anio"].between(2020, 2023))]
    .groupby(["cultivo", "anio"], observed=True)["produccion_ton"]
    .sum(min_count=1)
    .reset_index()
)

fig = px.line(
    puno_cultivo, x="anio", y="produccion_ton", color="cultivo", markers=True,
    labels={"produccion_ton": "Toneladas", "anio": "Año"},
)
# sin dtick=1 el eje (numerico, 4 puntos) generaba ticks fraccionarios (ej. 2021.5)
fig.update_xaxes(dtick=1, tickformat="d")
fig.update_layout(
    title=titulo_centrado("Puno — produccion anual por cultivo (mas representativos)"),
    margin=dict(t=90),
)
guardar(fig, "eda_puno_produccion_anual")
fig.show()


## 2.2 El Niño costero 2023–2024 — Piura, La Libertad, Ica

Anomalías de precipitación y producción respecto al promedio 2020–2022.

In [13]:
from plotly.subplots import make_subplots

costa = df[df["region"].isin(["Piura", "La Libertad", "Ica"])]
ref = costa[costa["anio"].between(2020, 2022)].groupby("region", observed=True)[["precipitacion", "temp_promedio"]].mean()
print("=== Referencia 2020–2022 (costa norte-centro) ===")
print(ref.round(3))

fig = make_subplots(rows=1, cols=2, subplot_titles=("Precipitación mensual", "Producción mensual"))
for region in ["Piura", "La Libertad", "Ica"]:
    g = costa[costa["region"] == region]
    m_precip = g.groupby("fecha", observed=True)["precipitacion"].mean()
    m_prod = g.groupby("fecha", observed=True)["produccion_piso_ton"].sum(min_count=1)
    fig.add_trace(go.Scatter(x=m_precip.index, y=m_precip.values, name=region,
                              legendgroup=region, line=dict(color=REGION_COLORS[region])), row=1, col=1)
    fig.add_trace(go.Scatter(x=m_prod.index, y=m_prod.values, name=region, showlegend=False,
                              legendgroup=region, line=dict(color=REGION_COLORS[region])), row=1, col=2)
for col in (1, 2):
    fig.add_vrect(x0="2023-01-01", x1="2024-12-31", fillcolor="coral", opacity=0.15, line_width=0, row=1, col=col)
fig.update_layout(
    title=titulo_centrado("Costa norte-centro — El Nino costero 2023-2024"),
    margin=dict(t=100),
)
guardar(fig, "eda_regional_nino_costa")
fig.show()


=== Referencia 2020–2022 (costa norte-centro) ===
             precipitacion  temp_promedio
region                                   
Ica                  0.223         19.099
La Libertad          0.737         17.841
Piura                0.503         23.185


### Profundización — magnitud del cambio respecto a la línea base

In [14]:
from plotly.subplots import make_subplots

evento_precip = costa[costa["anio"].isin([2023, 2024])].groupby("region", observed=True)["precipitacion"].mean()
base_precip = costa[costa["anio"].between(2020, 2022)].groupby("region", observed=True)["precipitacion"].mean()
evento_prod = (
    costa[costa["anio"].isin([2023, 2024])].groupby(["region", "anio"], observed=True)["produccion_piso_ton"]
    .sum(min_count=1).groupby("region").mean()
)
base_prod = (
    costa[costa["anio"].between(2020, 2022)].groupby(["region", "anio"], observed=True)["produccion_piso_ton"]
    .sum(min_count=1).groupby("region").mean()
)

# Subplot explicito 1x2: cada panel es una magnitud distinta (precipitacion,
# produccion) con su propio eje x (region) e y ("% cambio"), para que quede
# claro que son dos variables comparadas en paneles separados, no una sola serie.
# Solo las regiones costeras de "costa" (Piura, La Libertad, Ica) tienen datos aqui;
# San Martin/Junin/Puno no entraron al filtro y se omiten (antes salian como barras
# vacias por el reindex a REGION_ORDER completo).
REGION_ORDER_COSTA = [r for r in REGION_ORDER if r in costa["region"].unique()]
anomalia_precip = (100 * (evento_precip / base_precip - 1)).reindex(REGION_ORDER_COSTA)
anomalia_prod = (100 * (evento_prod / base_prod - 1)).reindex(REGION_ORDER_COSTA)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Precipitación (% cambio vs. 2020-2022)", "Producción (% cambio vs. 2020-2022)"],
    horizontal_spacing=0.12,
)
fig.add_trace(go.Bar(x=REGION_ORDER_COSTA, y=anomalia_precip,
                      marker_color=[REGION_COLORS[r] for r in REGION_ORDER_COSTA], showlegend=False),
              row=1, col=1)
fig.add_trace(go.Bar(x=REGION_ORDER_COSTA, y=anomalia_prod,
                      marker_color=[REGION_COLORS[r] for r in REGION_ORDER_COSTA], showlegend=False),
              row=1, col=2)
fig.update_yaxes(title_text="% cambio", row=1, col=1)
fig.update_yaxes(title_text="% cambio", row=1, col=2)
fig.update_xaxes(tickangle=0)
fig.update_layout(
    title=titulo_centrado("Anomalía % 2023-2024 vs. línea base 2020-2022"),
    width=1100, height=480, margin=dict(t=110),
)
guardar(fig, "eda_regional_nino_anomalia")
fig.show()

## 2.3 Verificación independiente — detección de anomalías (Isolation Forest)

¿Un método no supervisado, sin que se le diga dónde mirar, marca estos mismos periodos como
atípicos? Se aplica `IsolationForest` sobre clima + producción, estandarizados por z-score
**dentro de cada unidad** (así "anómalo" significa atípico para ese lugar, no solo "es Puno",
que ya es atípico frente a la costa). Esto **no es un modelo predictivo**: no entrena contra
un objetivo ni pronostica nada, solo mide qué tan fácil es aislar cada fila del resto de su
propio grupo — la misma familia de tarea no supervisada que el clustering.

De paso, el método encuentra un tercer episodio no analizado en este notebook: lluvias
intensas en Junín e Ica entre febrero y mayo de 2025, consistente con alertas de SENAMHI de
esas fechas. Su impacto en producción todavía no es observable por la cercanía del evento al
final de la serie.

In [15]:
import numpy as np
from sklearn.ensemble import IsolationForest

# Mismas 7 variables que en EXPERIMENTOS/anomalias_v1 (incluye temp_maxima,
# no solo CLIMA_EDA) para reproducir exactamente las cifras citadas en el informe.
FEATURES_ANOMALIA = ["temp_promedio", "temp_maxima", "precipitacion",
                     "humedad_relativa", "radiacion_solar", "humedad_suelo", "produccion_piso_ton"]
df_anom = df.dropna(subset=FEATURES_ANOMALIA).copy()

# Z-score dentro de cada unidad (region, piso, distrito): "anomalo" = atipico
# para ESE lugar, no solo "es Puno" (que ya es atipico frente a la costa).
grupo = df_anom.groupby("unidad")
for col in FEATURES_ANOMALIA:
    media = grupo[col].transform("mean")
    std = grupo[col].transform("std").replace(0, np.nan)
    df_anom[f"{col}_z"] = (df_anom[col] - media) / std
df_anom = df_anom.dropna(subset=[f"{c}_z" for c in FEATURES_ANOMALIA])

cols_z = [f"{c}_z" for c in FEATURES_ANOMALIA]
modelo = IsolationForest(contamination=0.05, random_state=42, n_estimators=200)
df_anom["anomalia"] = modelo.fit_predict(df_anom[cols_z]) == -1

tasa_base = df_anom["anomalia"].mean()

sequia_puno = (df_anom["region"] == "Puno") & (df_anom["anio"] == 2022)
nino_costa = (
    df_anom["region"].isin(["Ica", "La Libertad", "Piura", "San Martin"])
    & df_anom["anio"].isin([2023, 2024])
)
lluvias_2025 = (
    ((df_anom["region"] == "Junin") | ((df_anom["region"] == "Ica") & (df_anom["piso_ecologico"] == "costa")))
    & (df_anom["anio"] == 2025) & (df_anom["numero_mes"].between(2, 5))
)

tasa_puno = df_anom.loc[sequia_puno, "anomalia"].mean()
tasa_nino = df_anom.loc[nino_costa, "anomalia"].mean()
tasa_2025 = df_anom.loc[lluvias_2025, "anomalia"].mean()

print(f"Tasa base de anomalia (todo el dataset):        {tasa_base:.1%}  ({len(df_anom)} filas)")
print(f"Sequia Puno 2022:                                {tasa_puno:.1%}  ({sequia_puno.sum()} filas)")
print(f"El Nino costero 2023-2024:                       {tasa_nino:.1%}  ({nino_costa.sum()} filas)")
print(f"Lluvias Junin/Ica feb-may 2025:                   {tasa_2025:.1%}  ({lluvias_2025.sum()} filas)")


Tasa base de anomalia (todo el dataset):        5.0%  (2291 filas)
Sequia Puno 2022:                                15.0%  (40 filas)
El Nino costero 2023-2024:                       7.3%  (478 filas)
Lluvias Junin/Ica feb-may 2025:                   51.6%  (64 filas)


In [16]:
categorias = [
    "Resto del dataset<br>(línea base)", "Sequía Puno<br>2022",
    "El Niño costero<br>2023–2024", "Lluvias Junín/Ica<br>feb–may 2025",
]
valores = [tasa_base, tasa_puno, tasa_nino, tasa_2025]
colores = ["#B0B0B0", REGION_COLORS["Puno"], REGION_COLORS["Piura"], REGION_COLORS["Junin"]]

fig = go.Figure(go.Bar(
    x=valores, y=categorias, orientation="h", marker_color=colores,
    text=[f"{v:.1%}" for v in valores], textposition="outside",
))
fig.update_layout(
    title=titulo_centrado("Tasa de detección de anomalías por evento"),
    xaxis=dict(title="% de filas marcadas como anomalía (Isolation Forest)", tickformat=".0%",
               range=[0, max(valores) * 1.18]),
    width=750, height=380, margin=dict(l=170, r=40, t=90),
)
guardar(fig, "tasa_deteccion_eventos")
fig.show()


# Cierre

## Auditoría de calidad

- NaN en producción: meses sin cosecha reportada (política del pipeline, no imputados).
- **Siguiente paso:** `06_clustering_final.ipynb` construye tipologías sobre los perfiles más representativos.

In [17]:
# Auditoría NaN y resumen ejecutivo
clima_cols = [c for c in df.columns if c.startswith("temp_") or c in (
    "precipitacion", "humedad_relativa", "radiacion_solar", "humedad_suelo",
)]
nan_audit = df[clima_cols + ["produccion_piso_ton"]].isna().sum()
print("=== Auditoría NaN ===")
print(nan_audit[nan_audit > 0] if (nan_audit > 0).any() else "Sin NaN en columnas revisadas")

resumen = {
    "filas_regional": len(df),
    "filas_cultivo": len(df_cultivo),
    "unidades": df["unidad"].nunique(),
    "regiones": df["region"].nunique(),
    "nan_produccion": int(df["produccion_piso_ton"].isna().sum()),
    "figuras_generadas": len(list(RUTA_FIGURES.glob("eda_*.png"))),
}
print("\n=== Resumen ===")
for k, v in resumen.items():
    print(f"{k}: {v}")


=== Auditoría NaN ===
produccion_piso_ton    157
dtype: int64

=== Resumen ===
filas_regional: 2448
filas_cultivo: 2376
unidades: 34
regiones: 6
nan_produccion: 157
figuras_generadas: 13
